# 22.2 — Iterative deepening & bidirectional search

Repeat shallow DFS to get BFS-like depth guarantees, or search from both ends until the waves meet.

Iterative deepening recomputes shallow layers to keep active memory small. Bidirectional search shrinks depth by growing two reversible frontiers until they meet.

Save a copy to Drive to edit.

In [ ]:
from collections import deque
import math
import random
import matplotlib.pyplot as plt

SEED = 222
random.seed(SEED)


def line_graph(depth):
    graph = {}
    for i in range(depth):
        graph[i] = [i + 1]
    graph[depth] = []
    return graph


def reversible_tree(branching, depth):
    graph = {"": []}
    frontier = [""]
    for level in range(depth):
        new_frontier = []
        for node in frontier:
            for branch in range(branching):
                child = node + str(branch)
                graph.setdefault(node, []).append(child)
                graph.setdefault(child, []).append(node)
                new_frontier.append(child)
        frontier = new_frontier
    return graph, frontier[-1]


def grid_graph(width, height, missing_back_edges=None):
    missing_back_edges = set(missing_back_edges or [])
    graph = {}
    for r in range(height):
        for c in range(width):
            node = (r, c)
            graph[node] = []
            for dr, dc in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
                nxt = (r + dr, c + dc)
                if 0 <= nxt[0] < height and 0 <= nxt[1] < width:
                    if (nxt, node) not in missing_back_edges:
                        graph[node].append(nxt)
    return graph


def make_id_bidi_instances():
    graph1 = line_graph(4)
    graph2, goal2 = reversible_tree(3, 4)
    graph3 = grid_graph(6, 6)
    graph4, goal4 = reversible_tree(4, 5)
    one_way = {((3, 3), (3, 2)), ((3, 2), (3, 1)), ((2, 4), (2, 3))}
    graph5 = grid_graph(8, 8, missing_back_edges=one_way)
    return [
        {"name": "D1 depth-4 trace", "graph": graph1, "start": 0, "goal": 4, "reversible": True},
        {"name": "D2 wider reversible tree", "graph": graph2, "start": "", "goal": goal2, "reversible": True},
        {"name": "D3 deeper grid repeats", "graph": graph3, "start": (0, 0), "goal": (5, 5), "reversible": True},
        {"name": "D4 high-branching reversible", "graph": graph4, "start": "", "goal": goal4, "reversible": True},
        {"name": "D5 largest partly irreversible", "graph": graph5, "start": (0, 0), "goal": (7, 7), "reversible": False},
    ]


## The concept, built once (D1)

IDDFS runs depth-limited DFS with limit $L=0,1,2,\ldots$ until success. Bidirectional BFS uses the memory estimate $2b^{d/2}$ instead of one frontier near $b^d$, but only when predecessors are real.

In [ ]:
def depth_limited_dfs(graph, start, goal, limit):
    touched = 0
    stack = [(start, [start], 0)]
    seen_on_path = set()

    while stack:
        node, path, depth = stack.pop()
        touched = touched + 1
        if node == goal:
            return path, touched, True
        if depth < limit:
            for nxt in reversed(graph.get(node, [])):
                if nxt not in path:
                    stack.append((nxt, path + [nxt], depth + 1))

    return [], touched, False


def iddfs(graph, start, goal, max_depth):
    total_touched = 0
    per_limit = []
    for limit in range(max_depth + 1):
        path, touched, found = depth_limited_dfs(graph, start, goal, limit)
        total_touched = total_touched + touched
        per_limit.append(touched)
        if found:
            return {
                "path": path,
                "total_touched": total_touched,
                "per_limit": per_limit,
                "limit": limit,
            }
    return {
        "path": [],
        "total_touched": total_touched,
        "per_limit": per_limit,
        "limit": None,
    }


def predecessor_graph(graph):
    pred = {node: [] for node in graph}
    for node, nbrs in graph.items():
        for nbr in nbrs:
            pred.setdefault(nbr, []).append(node)
    return pred


def bidirectional_bfs(graph, start, goal, predecessors=None):
    predecessors = predecessors or predecessor_graph(graph)
    front = {start}
    back = {goal}
    parent_f = {start: None}
    parent_b = {goal: None}
    expanded = 0
    peak_memory = 2

    while front and back:
        if len(front) <= len(back):
            new_front = set()
            for node in front:
                expanded = expanded + 1
                for nxt in graph.get(node, []):
                    if nxt not in parent_f:
                        parent_f[nxt] = node
                        new_front.add(nxt)
                    if nxt in parent_b:
                        return {
                            "meet": nxt,
                            "expanded": expanded,
                            "peak_memory": peak_memory,
                        }
            front = new_front
        else:
            new_back = set()
            for node in back:
                expanded = expanded + 1
                for nxt in predecessors.get(node, []):
                    if nxt not in parent_b:
                        parent_b[nxt] = node
                        new_back.add(nxt)
                    if nxt in parent_f:
                        return {
                            "meet": nxt,
                            "expanded": expanded,
                            "peak_memory": peak_memory,
                        }
            back = new_back
        peak_memory = max(peak_memory, len(front) + len(back))

    return {
        "meet": None,
        "expanded": expanded,
        "peak_memory": peak_memory,
    }


On the lesson trace, limit $3$ fails and limit $4$ succeeds with $5-1=4$ moves. The exact touches are $1+3+6+8+15=33$. For $b=4,d=8$, one frontier stores $4^8=65{,}536$, while two frontiers store $2\cdot4^4=512$.

In [ ]:
lesson_per_limit = [1, 3, 6, 8, 15]
lesson_total = sum(lesson_per_limit)
line = line_graph(4)
limit3 = depth_limited_dfs(line, 0, 4, 3)
limit4 = depth_limited_dfs(line, 0, 4, 4)
one_frontier = 4 ** 8
two_frontiers = 2 * (4 ** 4)

assert limit3[2] is False
assert limit4[2] is True
assert len(limit4[0]) - 1 == 4
assert lesson_total == 33
assert lesson_per_limit[-1] == 15
assert one_frontier == 65536
assert two_frontiers == 512


## The dataset ladder

The ladder grows from a depth-4 trace to trees, grids, and a D5 graph with one-way edges that make naive backward search unsafe.

In [ ]:
instances = make_id_bidi_instances()
for item in instances:
    edge_count = sum(len(v) for v in item["graph"].values())
    print(item["name"], "nodes", len(item["graph"]), "edges", edge_count, "reversible", item["reversible"])


In [ ]:
rows = []
for item in instances:
    max_depth = 10
    id_result = iddfs(item["graph"], item["start"], item["goal"], max_depth)
    if item["reversible"]:
        bi_result = bidirectional_bfs(item["graph"], item["start"], item["goal"])
    else:
        bi_result = {
            "meet": None,
            "expanded": math.inf,
            "peak_memory": math.inf,
        }
    rows.append({
        "rung": item["name"].split()[0],
        "iddfs_touched": id_result["total_touched"],
        "iddfs_depth": id_result["limit"],
        "bidi_expanded": bi_result["expanded"],
        "peak_memory": bi_result["peak_memory"],
        "path_length": len(id_result["path"]),
        "per_limit": id_result["per_limit"],
    })

for row in rows:
    print(row)


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, item, row in zip(axes[0], instances, rows):
    ax.set_title(item["name"].split()[0])
    levels = row["per_limit"]
    ax.bar(range(len(levels)), levels, color="lightblue")
    ax.set_xlabel("limit")
    ax.set_ylabel("touched")

labels = [row["rung"] for row in rows]
axes[1, 0].plot(labels, [row["iddfs_touched"] for row in rows], marker="o", label="IDDFS touched")
axes[1, 0].set_title("nodes touched vs rung")
finite_memory = [row["peak_memory"] if math.isfinite(row["peak_memory"]) else 0 for row in rows]
axes[1, 1].plot(labels, finite_memory, marker="o", color="purple")
axes[1, 1].set_title("bidirectional peak memory")
for ax in axes[1, 2:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on D5: imaginary predecessor actions

The $2b^{d/2}$ story assumes the backward frontier contains true predecessors. D5 removes selected reverse edges; the fix is to validate reversibility and fall back to IDDFS or one-frontier BFS if the action model is not reversible.

In [ ]:
d5 = instances[-1]
fake_predecessors = {node: list(nbrs) for node, nbrs in d5["graph"].items()}
wrong = bidirectional_bfs(d5["graph"], d5["start"], d5["goal"], fake_predecessors)
fixed = iddfs(d5["graph"], d5["start"], d5["goal"], 14)
print("naive meet", wrong["meet"])
print("fallback path length", len(fixed["path"]))
print("fallback touched", fixed["total_touched"])


## Evaluate it + Practice

- Main metric: nodes touched and peak frontier memory.
- No-skill baseline: plain BFS from the start.
- Cheap sanity check: rerun D1 and verify the hand-trace number from the lesson exactly.
- Ablation: disable the backward frontier and memory returns toward one-frontier behavior.
- Failure signals: a meeting node reached by an edge that does not exist in the forward graph.

Practice prompts:
1. Change one D3 obstacle, edge, or score parameter and predict which nodes change before running.
2. Add one more D5 deceptive branch and compare the metric table.
3. Write a one-sentence rule for when the pitfall would be dangerous in production.